In [1]:
!pip install transformers torch sentencepiece

  Obtaining dependency information for sentencepiece from https://files.pythonhosted.org/packages/2d/81/92df5673c067148c2545b1bfe49adfd775bcc3a169a047f5a0e6575ddaca/sentencepiece-0.2.1-cp312-cp312-win_amd64.whl.metadata
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   - -------------------------------------- 0.1/1.1 MB 650.2 kB/s eta 0:00:02
   ---------- ----------------------------- 0.3/1.1 MB 2.5 MB/s eta 0:00:01
   ------------------------------ --------- 0.8/1.1 MB 5.0 MB/s eta 0:00:01
   ---------------------------------------- 1.1/1.1 MB 5.6 MB/s eta 0:00:00



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
from transformers import pipeline
import pandas as pd

# 1. Načtení Zero-Shot modelu (toto je tvůj nový "mozek")
# Model je multilingvální, takže mu můžeš psát česky i anglicky
classifier = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")

# 3. Tvůj český mapping (ten, co už máš)
medical_mapping = {
    "General Practice and Family Medicine": [
        "všeobecné praktické lékařství", "vnitřní lékařství",
        "praktické lékařství pro děti a dorost", "všeobecná sestra", "pediatrie"
    ],
    "Dentistry and Oral Health": [
        "zubní lékařství", "ortodoncie", "dentální hygienistka",
        "zubní technik", "klinický logoped"
    ],
    "Orthopedics, Joints and Musculoskeletal Pain": [ # Oprava pro tvé koleno!
        "ortopedie a traumatologie pohybového ústrojí", "fyzioterapeut",
        "rehabilitační a fyzikální medicína", "traumatologie",
        "ergoterapeut", "tělovýchovné lékařství", "klinická osteologie"
    ],
    "Cardiology and Vascular System": [
        "kardiologie", "angiologie", "cévní chirurgie", " dětská kardiologie"
    ],
    "Pulmonology and Respiratory Issues": [
        "pneumologie a ftizeologie", "infekční lékařství"
    ],
    "Neurology and Brain Health": [
        "neurologie", "dětská neurologie"
    ],
    "Psychiatry, Psychology and Mental Health": [
        "psychiatrie", "klinický psycholog", "dětská a dorostová psychiatrie"
    ],
    "Dermatology and Skin Conditions": [
        "dermatovenerologie"
    ],
    "Gynecology, Obstetrics and Women's Health": [
        "gynekologie a porodnictví", "porodní asistentka",
        "gynekologie dětí a dospívajících"
    ],
    "Urology and Nephrology": [
        "urologie", "nefrologie"
    ],
    "Ophthalmology and Vision Care": [
        "oftalmologie", "optometrista"
    ],
    "ENT, Otolaryngology and Hearing": [
        "otorinolaryngologie a chirurgie hlavy a krku", "foniatrie"
    ],
    "Endocrinology, Diabetes and Metabolism": [
        "endokrinologie a diabetologie", "nutriční terapeut"
    ],
    "Oncology and Cancer Care": [
        "klinická onkologie", "radiační onkologie", "paliativní medicína"
    ],
    "Allergy and Immunology": [
        "alergologie a klinická imunologie"
    ],
    "Gastroenterology and Digestive System": [
        "gastroenterologie"
    ],
    "General Surgery and Urgent Care": [
        "chirurgie", "dětská chirurgie", "plastická chirurgie", "urgentní medicína"
    ],
    "Pain Management and Algeziology": [
        "algeziologie"
    ],
    "Radiology and Diagnostic Imaging": [
        "radiologie a zobrazovací metody", "nukleární medicína"
    ],
    "Rheumatology and Autoimmune Diseases": [
        "revmatologie"
    ]
}

# Nezapomeň aktualizovat i seznam labelů pro AI
labels_for_ai = list(medical_mapping.keys())

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
import pandas as pd

# 1. Načtení vyčištěného registru z Fáze 1
# Cesta musí odpovídat tomu, kam jsi soubor uložil (pravděpodobně data/processed)
try:
    df_registr = pd.read_csv('../data/processed/czech_registre_final.csv')
    print(f"Registr úspěšně načten! Obsahuje {len(df_registr)} lékařů.")
except FileNotFoundError:
    print("Soubor nebyl nalezen. Zkontroluj cestu nebo zda jsi v Notebooku 01 soubor uložil.")

# 2. Malá kontrola - podívej se, jak vypadají data (sloupce)
print("Dostupné sloupce v registru:", df_registr.columns.tolist())

Registr úspěšně načten! Obsahuje 76321 lékařů.
Dostupné sloupce v registru: ['MistoPoskytovaniId', 'Ico', 'DruhZarizeni', 'NazevCely', 'Obec', 'Ulice', 'CisloDomovniOrientacni', 'Psc', 'Kraj', 'Okres', 'OborPece', 'GPS', 'PoskytovatelTelefon', 'PoskytovatelEmail', 'PoskytovatelWeb', 'RozsahPece', 'Adresa', 'OborPece_List']


C:\Users\honzi\AppData\Local\Temp\ipykernel_13792\1607337444.py:6: DtypeWarning: Columns (0: PoskytovatelTelefon) have mixed types. Specify dtype option on import or set low_memory=False.
  df_registr = pd.read_csv('../data/processed/czech_registre_final.csv')


In [21]:
def najdi_lekare_podle_symptomu(user_text, top_n=5):
    # PŘIDÁVÁME HYPOTÉZU: To je klíč k přesnosti v češtině!
    # Říkáme modelu, v jakém vztahu je text k těm štítkům.
    hypothesis = "Tento pacient popisuje příznaky, které léčí obor {}."

    result = classifier(
        user_text,
        labels_for_ai,
        multi_label=False,
        hypothesis_template=hypothesis
    )

    best_category = result['labels'][0]
    confidence = result['scores'][0]
    # A) AI Klasifikace (Zero-Shot) - zůstává stejná
    result = classifier(user_text, labels_for_ai, multi_label=False)
    best_category = result['labels'][0]

    print(f"--- ANALÝZA SYMPTOMU ---")
    print(f"Vstup: {user_text}")
    print(f"Detekovaná kategorie: {best_category}")

    # B) Převod na české obory
    cz_specialties = medical_mapping.get(best_category, [])

    # C) FILTRACE REGISTRU
    # Hledáme shodu v tvém sloupci 'OborPece_List'
    mask = df_registr['OborPece_List'].str.lower().apply(
        lambda x: any(spec in str(x).lower() for spec in cz_specialties)
    )

    found_doctors = df_registr[mask]

    print(f"Nalezeno celkem {len(found_doctors)} lékařů v celé ČR.")
    print("-" * 30)

    # D) EXPORT (Tady byla chyba - teď opraveno na tvé sloupce)
    # Používáme 'NazevCely' místo Jméno/Příjmení
    print(f"Detekovaná kategorie: {best_category} (Jistota: {confidence:.2%})")
    return found_doctors[['NazevCely', 'Obec', 'Ulice', 'OborPece_List', 'PoskytovatelTelefon']].head(top_n)

# --- TEST ---
najdi_lekare_podle_symptomu("My knee popped")

--- ANALÝZA SYMPTOMU ---
Vstup: My knee popped
Detekovaná kategorie: Dermatology and Skin Conditions
Nalezeno celkem 1436 lékařů v celé ČR.
------------------------------
Detekovaná kategorie: Dermatology and Skin Conditions (Jistota: 11.31%)


,NazevCely,Obec,Ulice,OborPece_List,PoskytovatelTelefon
11,Red Clinic s.r.o.,Praha 7,Jankovcova,dermatovenerologie,NaN
138,Psychiatrická nemocnice Kosmonosy,Kosmonosy,Lípy,dermatovenerologie,+420326715700
151,Psychiatrická nemocnice Kosmonosy,Kosmonosy,Lípy,dermatovenerologie,+420326715700
164,Psychiatrická nemocnice Kosmonosy,Kosmonosy,Lípy,dermatovenerologie,+420326715700
181,Psychiatrická nemocnice Jihlava,Jihlava,Brněnská,dermatovenerologie,NaN
